# C5_public_proprietary_strategy: Public/censored proprietary-strategy evaluation

Public appendix notebook from the Bachelor's Thesis *Evaluación robusta de estrategias de inversión: backtesting, sobreajuste y validación estadística*.

This notebook is included for transparency/reproducibility. It was originally developed in Google Colab; paths may need to be adapted for local execution.


In [ ]:
# ============================================================
# VERSION PUBLICA / CENSURADA DEL MODULO C5


In [ ]:
# ============================================================
# Este archivo se incluye en el apendice del TFG.
# Se mantiene la infraestructura comun previa al modulo C5.
# La implementacion privada de C5 se sustituye por una version
# publica basada exclusivamente en rentabilidades observadas.


In [ ]:
# ============================================================

# -*- coding: utf-8 -*-
"""Block5.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1RkeDo7mN8zKfdWgYAPWs7vqGwApkEhd9

Instalación
"""

!pip -q install pandas numpy yfinance matplotlib lxml html5lib beautifulsoup4 requests


In [ ]:
# ============================================================
# CELDA 1 - MONTAR GOOGLE DRIVE Y CREAR CARPETAS


In [ ]:
# ============================================================

from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

# Carpeta base del TFG en Drive
TFG_BASE_DIR = "/content/drive/MyDrive/TFG"

# Crear carpetas necesarias
Path(f"{TFG_BASE_DIR}").mkdir(parents=True, exist_ok=True)
Path(f"{TFG_BASE_DIR}/data").mkdir(parents=True, exist_ok=True)
Path(f"{TFG_BASE_DIR}/data/cache").mkdir(parents=True, exist_ok=True)
Path(f"{TFG_BASE_DIR}/results").mkdir(parents=True, exist_ok=True)
Path(f"{TFG_BASE_DIR}/logs").mkdir(parents=True, exist_ok=True)

print("Google Drive montado.")
print(f"Carpeta base del TFG: {TFG_BASE_DIR}")
print("Carpetas creadas correctamente.")

"""Código completo del Bloque 0"""


In [ ]:
# ============================================================
# CAP 0 - MOTOR REPRODUCIBLE DE DATOS, BACKTESTING Y MÉTRICAS
# Versión Colab con reconstrucción histórica del S&P 100


In [ ]:
# ============================================================

from __future__ import annotations

import json
import math
import time
import warnings
import re
from dataclasses import dataclass, asdict
from datetime import datetime, timezone
from io import StringIO
from pathlib import Path
from typing import Iterable, List, Optional, Tuple

import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import requests


In [ ]:
# ============================================================
# 1. CONFIGURACIÓN GENERAL


In [ ]:
# ============================================================

@dataclass
class BacktestConfig:
    # Periodo de análisis.
    # Yahoo Finance usa end_date como fecha final EXCLUSIVA.
    start_date: str = "2021-01-01"
    end_date: str = "2026-01-01"

    # Datos
    data_source: str = "Yahoo Finance via yfinance"
    universe_name: str = "S&P 100"

    # Archivo manual si algún día tenemos una fuente histórica propia.
    membership_file: str = "data/sp100_membership.csv"

    # Archivo reconstruido automáticamente desde revisiones históricas de Wikipedia.
    reconstructed_membership_file: str = "data/sp100_membership_reconstructed_wikipedia.csv"
    snapshot_log_file: str = "data/sp100_wikipedia_snapshot_log.csv"

    # Reconstrucción histórica
    use_reconstructed_membership: bool = True
    force_rebuild_membership: bool = True
    membership_history_frequency: str = "MS"

    # Importante:
    # Si esto está en False, el código NO vuelve a una lista actual si falla la reconstrucción.
    # Así evitamos meter survivorship bias sin querer.
    allow_current_fallback: bool = False

    # Benchmark
    # SPY = proxy invertible del S&P 500 con precios ajustados.
    # ^GSPC = índice S&P 500 de precio, no total return.
    benchmark_ticker: str = "SPY"
    benchmark_name: str = "S&P 500 benchmark"

    # Backtest
    initial_capital: float = 1.0
    trading_days_per_year: int = 252

    # Costes
    transaction_cost: float = 0.001
    cost_sensitivity: Tuple[float, ...] = (0.0, 0.0005, 0.001, 0.0025)

    # Tasa libre de riesgo anual provisional.
    risk_free_rate_annual: float = 0.0225

    # Limpieza
    min_price_coverage: float = 0.80
    min_assets_required: int = 20

    # Estrategia
    # "static_buy_hold": compra al inicio el universo histórico inicial y no rebalancea.
    # "dynamic_equal_weight": usa el universo histórico correspondiente en cada rebalanceo.
    strategy_mode: str = "dynamic_equal_weight"
    rebalance_frequency: str = "M"

    # Reproducibilidad
    random_seed: int = 123
    output_dir: str = "results/block0_sp100_survivorship_corrected"


CONFIG = BacktestConfig()


In [ ]:
# ============================================================
# 2. UTILIDADES GENERALES


In [ ]:
# ============================================================

def ensure_dir(path: str | Path) -> Path:
    path = Path(path)
    path.mkdir(parents=True, exist_ok=True)
    return path


def now_utc_iso() -> str:
    return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")


def save_json(obj: dict, path: str | Path) -> None:
    path = Path(path)
    ensure_dir(path.parent)
    with path.open("w", encoding="utf-8") as f:
        json.dump(obj, f, indent=4, ensure_ascii=False, default=str)


def to_yfinance_ticker(ticker: str) -> str:
    """
    Yahoo Finance usa '-' en vez de '.' para clases de acciones.
    Ejemplo: BRK.B -> BRK-B.
    """
    return str(ticker).strip().replace(".", "-")


def from_yfinance_ticker(ticker: str) -> str:
    return str(ticker).strip().replace("-", ".")


In [ ]:
# ============================================================
# 3. UNIVERSO S&P 100 HISTÓRICO


In [ ]:
# ============================================================

WIKIPEDIA_API_URL = "https://en.wikipedia.org/w/api.php"
WIKIPEDIA_TITLE_SP100 = "S&P 100"


# Lista solo de emergencia.
# Por defecto NO se usa porque allow_current_fallback=False.
SP100_FALLBACK_TICKERS = [
    "AAPL", "ABBV", "ABT", "ACN", "ADBE", "AIG", "AMD", "AMGN", "AMT", "AMZN",
    "AVGO", "AXP", "BA", "BAC", "BK", "BKNG", "BLK", "BMY", "BRK.B", "C",
    "CAT", "CHTR", "CL", "CMCSA", "COF", "COP", "COST", "CRM", "CSCO", "CVS",
    "CVX", "DE", "DHR", "DIS", "DUK", "EMR", "EXC", "F", "FDX", "GD",
    "GE", "GILD", "GM", "GOOG", "GOOGL", "GS", "HD", "HON", "IBM", "INTC",
    "JNJ", "JPM", "KHC", "KO", "LIN", "LLY", "LMT", "LOW", "MA", "MCD",
    "MDLZ", "MDT", "MET", "META", "MMM", "MO", "MRK", "MS", "MSFT", "NEE",
    "NFLX", "NKE", "NVDA", "ORCL", "PEP", "PFE", "PG", "PM", "PYPL", "QCOM",
    "RTX", "SBUX", "SCHW", "SO", "SPG", "T", "TGT", "TMO", "TXN", "UNH",
    "UNP", "UPS", "USB", "V", "VZ", "WBA", "WFC", "WMT", "XOM"
]


def clean_ticker_symbol(x: str) -> Optional[str]:
    """
    Limpia símbolos extraídos de tablas HTML.
    """
    if pd.isna(x):
        return None

    s = str(x).strip()
    s = re.sub(r"\[.*?\]", "", s)
    s = s.replace("\xa0", " ")
    s = s.strip()

    if not s:
        return None

    bad_values = {"symbol", "ticker", "nan", "none", "company", "security"}
    if s.lower() in bad_values:
        return None

    s = re.sub(r"[^A-Za-z0-9\.\-]", "", s)

    if not s:
        return None

    return s.upper()


def wikipedia_api_get(
    params: dict,
    max_retries: int = 4,
    sleep_seconds: float = 0.8,
) -> dict:
    """
    Petición robusta a la API de Wikipedia.
    """
    headers = {
        "User-Agent": "TFG-Backtest-Research/1.0 academic Colab notebook"
    }

    last_error = None

    for attempt in range(max_retries):
        try:
            response = requests.get(
                WIKIPEDIA_API_URL,
                params=params,
                headers=headers,
                timeout=30,
            )

            response.raise_for_status()
            data = response.json()

            if "error" in data:
                raise RuntimeError(data["error"])

            return data

        except Exception as e:
            last_error = e
            time.sleep(sleep_seconds * (attempt + 1))

    raise RuntimeError(f"No se pudo consultar Wikipedia API. Último error: {last_error}")


def get_wikipedia_revision_at(date: pd.Timestamp) -> Tuple[int, str]:
    """
    Obtiene la revisión de la página S&P 100 existente en una fecha concreta.
    """
    date = pd.Timestamp(date)
    rvstart = date.strftime("%Y-%m-%dT23:59:59Z")

    params = {
        "action": "query",
        "format": "json",
        "formatversion": "2",
        "prop": "revisions",
        "titles": WIKIPEDIA_TITLE_SP100,
        "rvprop": "ids|timestamp",
        "rvlimit": "1",
        "rvdir": "older",
        "rvstart": rvstart,
    }

    data = wikipedia_api_get(params)
    pages = data.get("query", {}).get("pages", [])

    if not pages or "revisions" not in pages[0] or not pages[0]["revisions"]:
        raise RuntimeError(f"No se encontró revisión de Wikipedia para {date.date()}")

    rev = pages[0]["revisions"][0]

    return int(rev["revid"]), str(rev["timestamp"])


def get_wikipedia_revision_html(revid: int) -> str:
    """
    Convierte una revisión concreta a HTML mediante action=parse.
    """
    params = {
        "action": "parse",
        "format": "json",
        "oldid": str(revid),
        "prop": "text",
    }

    data = wikipedia_api_get(params)
    html = data.get("parse", {}).get("text", {}).get("*")

    if not html:
        raise RuntimeError(f"No se pudo extraer HTML para oldid={revid}")

    return html


def extract_sp100_tickers_from_html(html: str) -> List[str]:
    """
    Extrae los tickers de la tabla de constituyentes del S&P 100.

    Busca tablas con columna Symbol/Ticker.
    Acepta una tabla si encuentra aproximadamente entre 80 y 120 símbolos.
    """
    tables = pd.read_html(StringIO(html))
    candidates = []

    for table in tables:
        table = table.copy()

        if isinstance(table.columns, pd.MultiIndex):
            table.columns = [
                " ".join([str(x) for x in col if str(x) != "nan"]).strip()
                for col in table.columns
            ]
        else:
            table.columns = [str(c).strip() for c in table.columns]

        symbol_col = None

        for col in table.columns:
            col_lower = str(col).lower()
            if "symbol" in col_lower or "ticker" in col_lower:
                symbol_col = col
                break

        if symbol_col is None:
            continue

        tickers = []

        for value in table[symbol_col].tolist():
            ticker = clean_ticker_symbol(value)
            if ticker is not None:
                tickers.append(ticker)

        tickers = sorted(set(tickers))

        if 80 <= len(tickers) <= 120:
            candidates.append(tickers)

    if not candidates:
        raise RuntimeError("No se encontró una tabla válida de constituyentes del S&P 100.")

    candidates = sorted(candidates, key=lambda x: abs(len(x) - 100))

    return candidates[0]


def fetch_sp100_snapshot_from_wikipedia(date: pd.Timestamp) -> Tuple[List[str], dict]:
    """
    Descarga la composición del S&P 100 según la página de Wikipedia
    tal como estaba en la fecha indicada.
    """
    revid, revision_timestamp = get_wikipedia_revision_at(date)
    html = get_wikipedia_revision_html(revid)
    tickers = extract_sp100_tickers_from_html(html)

    metadata = {
        "snapshot_date": str(pd.Timestamp(date).date()),
        "wiki_revision_id": revid,
        "wiki_revision_timestamp": revision_timestamp,
        "n_tickers": len(tickers),
        "status": "ok",
    }

    return tickers, metadata


def make_snapshot_dates(start_date: str, end_date: str, freq: str) -> List[pd.Timestamp]:
    """
    Genera fechas de snapshot entre start_date y end_date.

    end_date se considera exclusiva, igual que en Yahoo Finance.
    """
    start = pd.Timestamp(start_date).normalize()
    end_exclusive = pd.Timestamp(end_date).normalize()
    end_inclusive = end_exclusive - pd.Timedelta(days=1)

    regular_dates = pd.date_range(start=start, end=end_inclusive, freq=freq)

    dates = sorted(set([start] + list(regular_dates)))
    dates = [pd.Timestamp(d).normalize() for d in dates if start <= d <= end_inclusive]

    return dates


def snapshots_to_membership(snapshot_rows: List[dict]) -> pd.DataFrame:
    """
    Convierte snapshots mensuales en intervalos de pertenencia.
    """
    snapshots = pd.DataFrame(snapshot_rows)

    if snapshots.empty:
        raise RuntimeError("No hay snapshots para construir membership.")

    snapshots["snapshot_date"] = pd.to_datetime(snapshots["snapshot_date"])
    snapshots = snapshots.sort_values("snapshot_date")

    interval_rows = []
    unique_dates = sorted(snapshots["snapshot_date"].unique())

    for i, date in enumerate(unique_dates):
        current = snapshots[snapshots["snapshot_date"] == date]
        tickers = sorted(current["ticker"].dropna().unique().tolist())

        if i < len(unique_dates) - 1:
            interval_end = pd.Timestamp(unique_dates[i + 1]) - pd.Timedelta(days=1)
        else:
            interval_end = pd.NaT

        for ticker in tickers:
            interval_rows.append({
                "ticker": ticker,
                "yf_ticker": to_yfinance_ticker(ticker),
                "start": pd.Timestamp(date),
                "end": interval_end,
                "source": "wikipedia_historical_revision_snapshot",
            })

    membership = pd.DataFrame(interval_rows)

    if membership.empty:
        raise RuntimeError("Membership histórico vacío.")

    return membership


def reconstruct_sp100_membership_from_wikipedia(
    config: BacktestConfig,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Reconstruye la composición histórica del S&P 100 usando revisiones históricas
    de Wikipedia.

    Esto evita usar una lista actual para todo el pasado.
    """
    snapshot_dates = make_snapshot_dates(
        config.start_date,
        config.end_date,
        config.membership_history_frequency,
    )

    print("Reconstruyendo S&P 100 histórico desde Wikipedia.")
    print(f"Snapshots: {len(snapshot_dates)} fechas ({config.membership_history_frequency})")

    snapshot_rows = []
    snapshot_log = []

    last_good_tickers = None

    for i, date in enumerate(snapshot_dates):
        print(f"  Snapshot {i + 1}/{len(snapshot_dates)}: {date.date()}")

        try:
            tickers, meta = fetch_sp100_snapshot_from_wikipedia(date)
            last_good_tickers = tickers

        except Exception as e:
            if last_good_tickers is None:
                raise RuntimeError(
                    f"Falló el primer snapshot ({date.date()}) y no hay composición anterior para arrastrar. "
                    f"Error: {e}"
                )

            tickers = last_good_tickers
            meta = {
                "snapshot_date": str(date.date()),
                "wiki_revision_id": None,
                "wiki_revision_timestamp": None,
                "n_tickers": len(tickers),
                "status": f"carried_forward_after_error: {type(e).__name__}: {e}",
            }

        for ticker in tickers:
            snapshot_rows.append({
                "snapshot_date": str(date.date()),
                "ticker": ticker,
            })

        snapshot_log.append(meta)

        time.sleep(0.25)

    membership = snapshots_to_membership(snapshot_rows)
    snapshot_log_df = pd.DataFrame(snapshot_log)

    ensure_dir("data")

    membership.to_csv(config.reconstructed_membership_file, index=False)
    snapshot_log_df.to_csv(config.snapshot_log_file, index=False)

    return membership, snapshot_log_df


def load_membership_schedule(config: BacktestConfig) -> Tuple[pd.DataFrame, str]:
    """
    Carga o reconstruye el membership histórico.

    Orden:
    1. Si existe data/sp100_membership.csv manual, lo usa.
    2. Si existe membership reconstruido y force_rebuild_membership=False, lo usa.
    3. Si no existe, reconstruye desde revisiones históricas de Wikipedia.
    4. Si todo falla y allow_current_fallback=False, lanza error.
    """
    manual_path = Path(config.membership_file)
    reconstructed_path = Path(config.reconstructed_membership_file)

    if manual_path.exists():
        membership = pd.read_csv(manual_path)

        required = {"ticker", "start", "end"}
        missing = required - set(membership.columns)

        if missing:
            raise ValueError(f"Faltan columnas en {manual_path}: {missing}")

        membership["ticker"] = membership["ticker"].astype(str).str.strip()
        membership["yf_ticker"] = membership["ticker"].map(to_yfinance_ticker)
        membership["start"] = pd.to_datetime(membership["start"], errors="coerce")
        membership["end"] = pd.to_datetime(membership["end"], errors="coerce")

        return membership, "manual_historical_membership_file"

    if (
        config.use_reconstructed_membership
        and reconstructed_path.exists()
        and not config.force_rebuild_membership
    ):
        membership = pd.read_csv(reconstructed_path)

        membership["ticker"] = membership["ticker"].astype(str).str.strip()
        membership["yf_ticker"] = membership["ticker"].map(to_yfinance_ticker)
        membership["start"] = pd.to_datetime(membership["start"], errors="coerce")
        membership["end"] = pd.to_datetime(membership["end"], errors="coerce")

        return membership, "cached_wikipedia_reconstructed_historical_membership"

    if config.use_reconstructed_membership:
        try:
            membership, snapshot_log_df = reconstruct_sp100_membership_from_wikipedia(config)
            return membership, "wikipedia_reconstructed_historical_membership"

        except Exception as e:
            if not config.allow_current_fallback:
                raise RuntimeError(
                    "No se pudo reconstruir la composición histórica del S&P 100 y "
                    "allow_current_fallback=False. No se usará lista actual para evitar survivorship bias. "
                    f"Error original: {e}"
                )

    if config.allow_current_fallback:
        tickers = sorted(SP100_FALLBACK_TICKERS)

        membership = pd.DataFrame({
            "ticker": tickers,
            "yf_ticker": [to_yfinance_ticker(t) for t in tickers],
            "start": pd.Timestamp("1900-01-01"),
            "end": pd.NaT,
            "source": "current_fallback",
        })

        return membership, "current_fallback_SURVIVORSHIP_BIASED"

    raise RuntimeError("No se pudo cargar ni reconstruir membership histórico.")


def active_tickers_on_date(membership: pd.DataFrame, date: pd.Timestamp) -> List[str]:
    """
    Devuelve los tickers activos en una fecha concreta según la tabla de pertenencia.
    """
    date = pd.Timestamp(date)

    active = membership[
        (membership["start"].isna() | (membership["start"] <= date)) &
        (membership["end"].isna() | (membership["end"] >= date))
    ]

    return sorted(active["yf_ticker"].dropna().unique().tolist())


def all_tickers_needed(membership: pd.DataFrame, start_date: str, end_date: str) -> List[str]:
    """
    Devuelve todos los tickers que podrían ser necesarios entre start_date y end_date.
    """
    start = pd.Timestamp(start_date)
    end = pd.Timestamp(end_date)

    relevant = membership[
        (membership["end"].isna() | (membership["end"] >= start)) &
        (membership["start"].isna() | (membership["start"] <= end))
    ]

    return sorted(relevant["yf_ticker"].dropna().unique().tolist())


In [ ]:
# ============================================================
# 4. DESCARGA Y PREPROCESADO DE DATOS


In [ ]:
# ============================================================

def download_adjusted_close(
    tickers: Iterable[str],
    start_date: str,
    end_date: str,
    chunk_size: int = 25,
    sleep_seconds: float = 1.0,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Descarga precios ajustados mediante yfinance.

    Con auto_adjust=True, la columna Close queda ajustada.
    Descarga por bloques para reducir errores en Colab/Yahoo.
    """
    tickers = sorted(set([to_yfinance_ticker(t) for t in tickers]))

    if not tickers:
        raise ValueError("No hay tickers para descargar.")

    all_prices = []
    report_rows = []

    print(f"Descargando {len(tickers)} tickers desde Yahoo Finance...")

    for i in range(0, len(tickers), chunk_size):
        chunk = tickers[i:i + chunk_size]
        print(f"  Bloque {i // chunk_size + 1}: {len(chunk)} tickers")

        try:
            data = yf.download(
                tickers=chunk,
                start=start_date,
                end=end_date,
                auto_adjust=True,
                progress=False,
                group_by="ticker",
                threads=True,
            )

        except Exception as e:
            for t in chunk:
                report_rows.append({
                    "yf_ticker": t,
                    "status": f"download_exception: {type(e).__name__}",
                    "coverage": 0.0,
                    "first_valid_date": None,
                    "last_valid_date": None,
                })

            time.sleep(sleep_seconds)
            continue

        prices_chunk = pd.DataFrame()

        if len(chunk) == 1:
            t = chunk[0]

            if isinstance(data, pd.DataFrame) and "Close" in data.columns:
                prices_chunk[t] = data["Close"]

        else:
            for t in chunk:
                try:
                    if isinstance(data.columns, pd.MultiIndex):
                        if t in data.columns.get_level_values(0):
                            close = data[t]["Close"].copy()
                            prices_chunk[t] = close
                    else:
                        if "Close" in data.columns:
                            prices_chunk[t] = data["Close"]

                except Exception:
                    pass

        if not prices_chunk.empty:
            prices_chunk = prices_chunk.sort_index()
            prices_chunk.index.name = "Date"
            prices_chunk = prices_chunk.dropna(axis=1, how="all")
            all_prices.append(prices_chunk)

        available = set(prices_chunk.columns) if not prices_chunk.empty else set()

        for t in chunk:
            if t in available:
                s = prices_chunk[t]
                coverage = float(s.notna().mean())
                first_date = s.first_valid_index()
                last_date = s.last_valid_index()
                status = "ok"
            else:
                coverage = 0.0
                first_date = None
                last_date = None
                status = "download_failed_or_no_close"

            report_rows.append({
                "yf_ticker": t,
                "status": status,
                "coverage": coverage,
                "first_valid_date": first_date,
                "last_valid_date": last_date,
            })

        time.sleep(sleep_seconds)

    if not all_prices:
        raise RuntimeError("No se han podido descargar precios. Revisa conexión, tickers o yfinance.")

    prices = pd.concat(all_prices, axis=1)
    prices = prices.loc[:, ~prices.columns.duplicated()]
    prices = prices.sort_index()
    prices.index.name = "Date"

    report = pd.DataFrame(report_rows)

    return prices, report


def clean_prices(
    prices: pd.DataFrame,
    min_coverage: float,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Limpieza conservadora.
    """
    prices = prices.copy()
    prices = prices[~prices.index.duplicated(keep="first")]
    prices = prices.sort_index()

    coverage = prices.notna().mean()

    keep = coverage[coverage >= min_coverage].index.tolist()
    excluded = coverage[coverage < min_coverage].index.tolist()

    exclusions = pd.DataFrame({
        "yf_ticker": excluded,
        "reason": f"coverage_below_{min_coverage:.0%}",
        "coverage": coverage.loc[excluded].values if len(excluded) > 0 else [],
    })

    clean = prices[keep].copy()
    clean = clean.ffill()

    return clean, exclusions


def compute_simple_returns(prices: pd.DataFrame) -> pd.DataFrame:
    returns = prices.pct_change(fill_method=None)
    returns = returns.replace([np.inf, -np.inf], np.nan)

    return returns


In [ ]:
# ============================================================
# 5. BACKTESTS BASE


In [ ]:
# ============================================================

def backtest_static_buy_hold(
    prices: pd.DataFrame,
    membership: pd.DataFrame,
    config: BacktestConfig,
    transaction_cost: Optional[float] = None,
) -> Tuple[pd.Series, pd.Series, pd.DataFrame, dict]:
    """
    Buy & Hold estático:
    - Toma los miembros activos al inicio del backtest.
    - Compra cartera equally weighted.
    - No rebalancea.
    """
    transaction_cost = config.transaction_cost if transaction_cost is None else transaction_cost

    prices = prices.copy().dropna(axis=0, how="all")
    first_date = prices.index[0]

    active = active_tickers_on_date(membership, first_date)
    active = [t for t in active if t in prices.columns]

    first_prices = prices.loc[first_date, active].dropna()
    active = first_prices.index.tolist()

    if len(active) < config.min_assets_required:
        raise RuntimeError(
            f"Activos disponibles insuficientes para Buy & Hold: {len(active)} "
            f"(mínimo {config.min_assets_required})."
        )

    n = len(active)
    gross_weights = pd.Series(1.0 / n, index=active)

    entry_turnover = gross_weights.abs().sum()
    initial_cost = transaction_cost * entry_turnover * config.initial_capital
    investable_capital = config.initial_capital - initial_cost

    shares = (investable_capital * gross_weights) / first_prices

    portfolio_values = prices[active].ffill().mul(shares, axis=1).sum(axis=1)
    equity = portfolio_values.rename("strategy_equity")

    equity.iloc[0] = investable_capital

    returns = equity.pct_change(fill_method=None)
    returns.iloc[0] = equity.iloc[0] / config.initial_capital - 1.0
    returns = returns.rename("strategy_returns")

    weights = pd.DataFrame(0.0, index=prices.index, columns=active)
    weights.loc[:, active] = gross_weights.values

    metadata = {
        "strategy": "static_buy_hold",
        "first_date": str(first_date.date()),
        "n_assets": n,
        "transaction_cost": transaction_cost,
        "entry_turnover": float(entry_turnover),
        "initial_cost": float(initial_cost),
        "active_tickers": active,
    }

    return returns, equity, weights, metadata


def backtest_dynamic_equal_weight(
    prices: pd.DataFrame,
    membership: pd.DataFrame,
    config: BacktestConfig,
    transaction_cost: Optional[float] = None,
) -> Tuple[pd.Series, pd.Series, pd.DataFrame, dict]:
    """
    Equal-weight dinámico:
    - En cada fecha de rebalanceo selecciona miembros activos según membership histórico.
    - Rebalancea a pesos iguales.
    """
    transaction_cost = config.transaction_cost if transaction_cost is None else transaction_cost

    prices = prices.copy().dropna(axis=0, how="all").ffill()
    returns_assets = compute_simple_returns(prices).fillna(0.0)

    dates = prices.index

    rebalance_dates_raw = prices.resample(config.rebalance_frequency).last().index
    rebalance_dates = []

    for d in rebalance_dates_raw:
        idx = prices.index.get_indexer([d], method="pad")[0]

        if idx >= 0:
            rebalance_dates.append(prices.index[idx])

    rebalance_dates = sorted(set([d for d in rebalance_dates if d in dates]))

    all_cols = prices.columns.tolist()

    weights = pd.DataFrame(0.0, index=dates, columns=all_cols)

    current_weights = pd.Series(0.0, index=all_cols)
    prev_weights = pd.Series(0.0, index=all_cols)

    equity = pd.Series(index=dates, dtype=float, name="strategy_equity")
    strategy_returns = pd.Series(index=dates, dtype=float, name="strategy_returns")

    capital = config.initial_capital
    rebalance_count = 0
    total_turnover = 0.0

    for i, date in enumerate(dates):
        cost = 0.0

        if i == 0 or date in rebalance_dates:
            active = active_tickers_on_date(membership, date)
            active = [t for t in active if t in all_cols and not pd.isna(prices.loc[date, t])]

            if len(active) >= config.min_assets_required:
                target_weights = pd.Series(0.0, index=all_cols)
                target_weights.loc[active] = 1.0 / len(active)
            else:
                target_weights = prev_weights.copy()

            turnover = float((target_weights - prev_weights).abs().sum())
            cost = transaction_cost * turnover

            total_turnover += turnover
            rebalance_count += 1

            current_weights = target_weights.copy()
            prev_weights = current_weights.copy()

        if i == 0:
            r = -cost
        else:
            r_gross = float(current_weights.fillna(0.0).dot(returns_assets.iloc[i].fillna(0.0)))
            r = r_gross - cost

        capital *= (1.0 + r)

        equity.loc[date] = capital
        strategy_returns.loc[date] = r
        weights.loc[date] = current_weights.values

    metadata = {
        "strategy": "dynamic_equal_weight",
        "rebalance_frequency": config.rebalance_frequency,
        "transaction_cost": transaction_cost,
        "rebalance_count": rebalance_count,
        "total_turnover": total_turnover,
        "average_turnover_per_rebalance": total_turnover / rebalance_count if rebalance_count else np.nan,
    }

    return strategy_returns, equity, weights, metadata


def backtest_benchmark(
    benchmark_prices: pd.Series,
    config: BacktestConfig,
) -> Tuple[pd.Series, pd.Series]:
    """
    Benchmark Buy & Hold.
    No aplicamos costes al benchmark por defecto.
    """
    px = benchmark_prices.dropna().copy()
    px = px / px.iloc[0] * config.initial_capital
    px.name = "benchmark_equity"

    ret = px.pct_change(fill_method=None)
    ret.iloc[0] = 0.0
    ret.name = "benchmark_returns"

    return ret, px


In [ ]:
# ============================================================
# 6. MÉTRICAS


In [ ]:
# ============================================================

def format_index_endpoint(x):
    """
    Convierte un índice a string de forma robusta.
    """
    if hasattr(x, "date"):
        return str(x.date())

    return str(x)


def max_drawdown(equity: pd.Series) -> float:
    equity = equity.dropna()

    if len(equity) == 0:
        return np.nan

    running_max = equity.cummax()
    dd = equity / running_max - 1.0

    return float(dd.min())


def drawdown_series(equity: pd.Series) -> pd.Series:
    equity = equity.dropna()

    if len(equity) == 0:
        return pd.Series(dtype=float, name="drawdown")

    running_max = equity.cummax()

    return (equity / running_max - 1.0).rename("drawdown")


def annualized_volatility(
    returns: pd.Series,
    trading_days: int = 252,
) -> float:
    returns = returns.dropna()

    if len(returns) < 2:
        return np.nan

    return float(returns.std(ddof=1) * math.sqrt(trading_days))


def cagr_from_equity(
    equity: pd.Series,
    initial_capital: float,
    trading_days: int = 252,
) -> float:
    equity = equity.dropna()

    if len(equity) < 2:
        return np.nan

    years = len(equity) / trading_days
    final_value = equity.iloc[-1]

    if final_value <= 0:
        return -1.0

    return float((final_value / initial_capital) ** (1.0 / years) - 1.0)


def compute_metrics(
    returns: pd.Series,
    equity: pd.Series,
    config: BacktestConfig,
    label: str,
) -> pd.Series:
    returns = returns.dropna()
    equity = equity.dropna()

    if len(returns) == 0 or len(equity) == 0:
        return pd.Series({
            "label": label,
            "start": None,
            "end": None,
            "n_observations": 0,
            "cumulative_return": np.nan,
            "CAGR": np.nan,
            "annualized_volatility": np.nan,
            "Sharpe": np.nan,
            "Sortino": np.nan,
            "max_drawdown": np.nan,
            "Calmar": np.nan,
            "hit_ratio": np.nan,
            "risk_free_rate_annual": config.risk_free_rate_annual,
        })

    rf_daily = (1.0 + config.risk_free_rate_annual) ** (1.0 / config.trading_days_per_year) - 1.0
    excess = returns - rf_daily

    cumulative_return = float(equity.iloc[-1] / config.initial_capital - 1.0)

    cagr = cagr_from_equity(
        equity,
        config.initial_capital,
        config.trading_days_per_year,
    )

    vol = annualized_volatility(
        returns,
        config.trading_days_per_year,
    )

    if len(excess) > 1 and excess.std(ddof=1) > 0:
        sharpe = float(
            excess.mean() / excess.std(ddof=1) * math.sqrt(config.trading_days_per_year)
        )
    else:
        sharpe = np.nan

    downside = excess[excess < 0]

    if len(downside) > 1 and downside.std(ddof=1) > 0:
        sortino = float(
            excess.mean() / downside.std(ddof=1) * math.sqrt(config.trading_days_per_year)
        )
    else:
        sortino = np.nan

    mdd = max_drawdown(equity)

    if not np.isnan(mdd) and mdd < 0:
        calmar = float(cagr / abs(mdd))
    else:
        calmar = np.nan

    hit_ratio = float((returns > 0).mean())

    return pd.Series({
        "label": label,
        "start": format_index_endpoint(equity.index[0]),
        "end": format_index_endpoint(equity.index[-1]),
        "n_observations": int(len(returns)),
        "cumulative_return": cumulative_return,
        "CAGR": cagr,
        "annualized_volatility": vol,
        "Sharpe": sharpe,
        "Sortino": sortino,
        "max_drawdown": mdd,
        "Calmar": calmar,
        "hit_ratio": hit_ratio,
        "risk_free_rate_annual": config.risk_free_rate_annual,
    })


def compute_turnover(weights: pd.DataFrame) -> pd.Series:
    if weights.empty:
        return pd.Series(dtype=float, name="turnover")

    turnover = weights.diff().abs().sum(axis=1)
    turnover.iloc[0] = weights.iloc[0].abs().sum()

    return turnover.rename("turnover")


In [ ]:
# ============================================================
# 7. BOOTSTRAP SIMPLE


In [ ]:
# ============================================================

def bootstrap_metrics_simple(
    returns: pd.Series,
    config: BacktestConfig,
    n_bootstrap: int = 1000,
) -> pd.DataFrame:
    """
    Bootstrap simple sobre rentabilidades.
    Más adelante podemos sustituirlo por block bootstrap.
    """
    rng = np.random.default_rng(config.random_seed)
    r = returns.dropna().values

    rows = []

    for b in range(n_bootstrap):
        sample = rng.choice(r, size=len(r), replace=True)
        equity = pd.Series(config.initial_capital * np.cumprod(1.0 + sample))
        sample_returns = pd.Series(sample)

        m = compute_metrics(sample_returns, equity, config, label=f"bootstrap_{b}")
        rows.append(m)

    return pd.DataFrame(rows)


In [ ]:
# ============================================================
# 8. FIGURAS


In [ ]:
# ============================================================

def plot_equity_curves(
    strategy_equity: pd.Series,
    benchmark_equity: pd.Series,
    output_path: str | Path,
) -> None:
    plt.figure(figsize=(10, 6))
    strategy_equity.dropna().plot(label="Strategy")
    benchmark_equity.dropna().plot(label="Benchmark")
    plt.title("Curva de capital")
    plt.xlabel("Fecha")
    plt.ylabel("Capital normalizado")
    plt.legend()
    plt.tight_layout()
    plt.savefig(output_path, dpi=160)
    plt.close()


def plot_drawdowns(
    strategy_equity: pd.Series,
    benchmark_equity: pd.Series,
    output_path: str | Path,
) -> None:
    plt.figure(figsize=(10, 6))
    drawdown_series(strategy_equity).plot(label="Strategy")
    drawdown_series(benchmark_equity).plot(label="Benchmark")
    plt.title("Drawdown")
    plt.xlabel("Fecha")
    plt.ylabel("Drawdown")
    plt.legend()
    plt.tight_layout()
    plt.savefig(output_path, dpi=160)
    plt.close()


In [ ]:
# ============================================================
# 9. PIPELINE PRINCIPAL


In [ ]:
# ============================================================

def run_block0(config: BacktestConfig = CONFIG) -> None:
    warnings.filterwarnings("ignore", category=FutureWarning)
    np.random.seed(config.random_seed)

    out = ensure_dir(config.output_dir)
    ensure_dir(out / "figures")
    ensure_dir("data/raw")
    ensure_dir("data/processed")
    ensure_dir("logs")

    run_metadata = {
        "run_timestamp_utc": now_utc_iso(),
        "config": asdict(config),
    }

    save_json(run_metadata, out / "config_run.json")

    # Universo histórico
    membership, membership_source = load_membership_schedule(config)
    run_metadata["membership_source"] = membership_source

    tickers_needed = all_tickers_needed(
        membership,
        config.start_date,
        config.end_date,
    )

    all_download_tickers = sorted(set(tickers_needed + [config.benchmark_ticker]))

    # Descarga
    prices_raw, download_report = download_adjusted_close(
        all_download_tickers,
        config.start_date,
        config.end_date,
    )

    prices_raw.to_csv("data/raw/prices_raw.csv")
    download_report.to_csv("logs/download_report.csv", index=False)

    if config.benchmark_ticker not in prices_raw.columns:
        raise RuntimeError(
            f"No se pudo descargar el benchmark {config.benchmark_ticker}. "
            f"Prueba con '^GSPC' o revisa conexión/yfinance."
        )

    benchmark_prices = prices_raw[config.benchmark_ticker].dropna()

    asset_prices_raw = prices_raw.drop(columns=[config.benchmark_ticker], errors="ignore")
    asset_prices, exclusions = clean_prices(asset_prices_raw, config.min_price_coverage)

    asset_prices.to_csv("data/processed/prices_clean.csv")
    compute_simple_returns(asset_prices).to_csv("data/processed/returns.csv")
    exclusions.to_csv("logs/exclusions.csv", index=False)

    # Backtest de estrategia
    if config.strategy_mode == "static_buy_hold":
        strategy_returns, strategy_equity, weights, strategy_meta = backtest_static_buy_hold(
            asset_prices,
            membership,
            config,
        )

    elif config.strategy_mode == "dynamic_equal_weight":
        strategy_returns, strategy_equity, weights, strategy_meta = backtest_dynamic_equal_weight(
            asset_prices,
            membership,
            config,
        )

    else:
        raise ValueError(f"strategy_mode no reconocido: {config.strategy_mode}")

    # Benchmark
    benchmark_returns, benchmark_equity = backtest_benchmark(
        benchmark_prices,
        config,
    )

    # Alinear fechas
    common_dates = strategy_equity.index.intersection(benchmark_equity.index)

    strategy_equity = strategy_equity.loc[common_dates]
    strategy_returns = strategy_returns.loc[common_dates]

    benchmark_equity = benchmark_equity.loc[common_dates]
    benchmark_returns = benchmark_returns.loc[common_dates]

    # Métricas principales
    strategy_metrics = compute_metrics(
        strategy_returns,
        strategy_equity,
        config,
        "SP100_strategy",
    )

    benchmark_metrics = compute_metrics(
        benchmark_returns,
        benchmark_equity,
        config,
        config.benchmark_name,
    )

    metrics = pd.DataFrame([strategy_metrics, benchmark_metrics])

    # Turnover
    turnover = compute_turnover(weights)

    # Sensibilidad a costes
    sensitivity_rows = []

    for cost in config.cost_sensitivity:
        if config.strategy_mode == "static_buy_hold":
            r_s, eq_s, _, _ = backtest_static_buy_hold(
                asset_prices,
                membership,
                config,
                transaction_cost=cost,
            )

        else:
            r_s, eq_s, _, _ = backtest_dynamic_equal_weight(
                asset_prices,
                membership,
                config,
                transaction_cost=cost,
            )

        eq_s = eq_s.loc[eq_s.index.intersection(common_dates)]
        r_s = r_s.loc[eq_s.index]

        m = compute_metrics(r_s, eq_s, config, f"strategy_cost_{cost}")
        m["transaction_cost"] = cost
        sensitivity_rows.append(m)

    sensitivity = pd.DataFrame(sensitivity_rows)

    # Bootstrap simple
    boot = bootstrap_metrics_simple(
        strategy_returns,
        config,
        n_bootstrap=1000,
    )

    boot_summary = boot[["CAGR", "Sharpe", "Sortino", "max_drawdown", "Calmar"]].quantile(
        [0.025, 0.05, 0.50, 0.95, 0.975]
    )

    # Exportar outputs
    strategy_returns.to_csv(out / "strategy_returns.csv")
    strategy_equity.to_csv(out / "strategy_equity.csv")

    benchmark_returns.to_csv(out / "benchmark_returns.csv")
    benchmark_equity.to_csv(out / "benchmark_equity.csv")

    weights.to_csv(out / "weights.csv")
    turnover.to_csv(out / "turnover.csv")

    metrics.to_csv(out / "metrics.csv", index=False)
    sensitivity.to_csv(out / "cost_sensitivity.csv", index=False)

    boot.to_csv(out / "bootstrap_metrics.csv", index=False)
    boot_summary.to_csv(out / "bootstrap_summary.csv")

    # Metadata
    run_metadata["strategy_metadata"] = strategy_meta
    run_metadata["n_raw_assets_downloaded"] = len(tickers_needed)
    run_metadata["n_assets_after_cleaning"] = int(asset_prices.shape[1])
    run_metadata["n_excluded_assets"] = int(len(exclusions))
    run_metadata["benchmark_ticker"] = config.benchmark_ticker

    if "SURVIVORSHIP_BIASED" in membership_source:
        membership_warning = "WARNING: current/fallback constituents used. Survivorship bias present."
    elif "wikipedia" in membership_source.lower():
        membership_warning = (
            "Historical public reconstruction from Wikipedia page revisions. "
            "This avoids using future constituents, but is not the official licensed S&P DJI constituent history."
        )
    else:
        membership_warning = "Historical membership file used."

    run_metadata["membership_warning"] = membership_warning

    save_json(run_metadata, out / "metadata.json")

    # Figuras
    plot_equity_curves(
        strategy_equity,
        benchmark_equity,
        out / "figures/equity_curves.png",
    )

    plot_drawdowns(
        strategy_equity,
        benchmark_equity,
        out / "figures/drawdowns.png",
    )

    # Resumen por pantalla
    print("\n=== Bloque 0 completado ===")
    print(f"Modo estrategia: {config.strategy_mode}")
    print(f"Fuente membership: {membership_source}")
    print(f"Activos tras limpieza: {asset_prices.shape[1]}")
    print(f"Benchmark: {config.benchmark_ticker}")
    print(f"Outputs guardados en: {out.resolve()}")

    print("\nMétricas principales:")
    with pd.option_context("display.max_columns", None, "display.width", 160):
        print(metrics)


print("Bloque 0 cargado correctamente. Ahora cambia el Bloque 3 y ejecuta el Bloque 4.")


In [ ]:
# ============================================================
# BLOQUE 2.5 - CACHÉ DE MEMBERSHIP Y DATOS DE MERCADO


In [ ]:
# ============================================================

import hashlib


def make_market_data_cache_key(
    tickers: list,
    start_date: str,
    end_date: str,
    data_source: str,
) -> str:
    """
    Genera una clave única para identificar una descarga de datos.

    Si cambias:
    - tickers,
    - start_date,
    - end_date,
    - fuente de datos,

    se genera una caché distinta.
    """
    raw = {
        "tickers": sorted(tickers),
        "start_date": start_date,
        "end_date": end_date,
        "data_source": data_source,
    }

    raw_string = json.dumps(raw, sort_keys=True)
    return hashlib.md5(raw_string.encode("utf-8")).hexdigest()[:12]


def load_or_download_market_data(
    tickers: list,
    config: BacktestConfig,
) -> Tuple[pd.DataFrame, pd.DataFrame, str]:
    """
    Carga precios desde caché si existen.

    Si no existen o si CONFIG.force_redownload_market_data=True,
    descarga desde Yahoo Finance y guarda caché.
    """
    cache_dir = ensure_dir(getattr(config, "market_data_cache_dir", "data/cache"))

    cache_key = make_market_data_cache_key(
        tickers=tickers,
        start_date=config.start_date,
        end_date=config.end_date,
        data_source=config.data_source,
    )

    prices_cache_path = cache_dir / f"prices_{cache_key}.csv"
    report_cache_path = cache_dir / f"download_report_{cache_key}.csv"
    metadata_cache_path = cache_dir / f"market_data_metadata_{cache_key}.json"

    use_cached_market_data = getattr(config, "use_cached_market_data", True)
    force_redownload_market_data = getattr(config, "force_redownload_market_data", False)

    if (
        use_cached_market_data
        and not force_redownload_market_data
        and prices_cache_path.exists()
        and report_cache_path.exists()
    ):
        print(f"Cargando datos de mercado desde caché: {prices_cache_path}")

        prices = pd.read_csv(
            prices_cache_path,
            index_col=0,
            parse_dates=True,
        )

        prices.index.name = "Date"
        prices = prices.sort_index()

        report = pd.read_csv(report_cache_path)

        return prices, report, "market_data_cache"

    print("No hay caché válida de datos de mercado. Descargando desde Yahoo Finance...")

    prices, report = download_adjusted_close(
        tickers,
        config.start_date,
        config.end_date,
    )

    prices.to_csv(prices_cache_path)
    report.to_csv(report_cache_path, index=False)

    metadata = {
        "created_at_utc": now_utc_iso(),
        "cache_key": cache_key,
        "start_date": config.start_date,
        "end_date": config.end_date,
        "data_source": config.data_source,
        "n_tickers_requested": len(tickers),
        "tickers": sorted(tickers),
        "prices_cache_path": str(prices_cache_path),
        "report_cache_path": str(report_cache_path),
    }

    save_json(metadata, metadata_cache_path)

    print(f"Datos guardados en caché: {prices_cache_path}")

    return prices, report, "market_data_downloaded"


def run_block0(config: BacktestConfig = CONFIG) -> None:
    """
    Pipeline principal con caché.

    Cambios respecto al anterior:
    - Puede reutilizar membership histórico reconstruido.
    - Puede reutilizar precios descargados de Yahoo Finance.
    - Evita repetir los 6 minutos de descarga/reconstrucción cada vez.
    """
    warnings.filterwarnings("ignore", category=FutureWarning)
    np.random.seed(config.random_seed)

    out = ensure_dir(config.output_dir)
    ensure_dir(out / "figures")
    ensure_dir("data/raw")
    ensure_dir("data/processed")
    ensure_dir("logs")
    ensure_dir("data/cache")

    run_metadata = {
        "run_timestamp_utc": now_utc_iso(),
        "config": asdict(config),
    }

    # Guardamos también atributos añadidos dinámicamente en Colab.
    dynamic_config_attrs = [
        "use_cached_market_data",
        "force_redownload_market_data",
        "market_data_cache_dir",
    ]

    for attr in dynamic_config_attrs:
        if hasattr(config, attr):
            run_metadata["config"][attr] = getattr(config, attr)

    save_json(run_metadata, out / "config_run.json")

    # ========================================================
    # 1. Universo histórico
    # ========================================================

    membership, membership_source = load_membership_schedule(config)
    run_metadata["membership_source"] = membership_source

    tickers_needed = all_tickers_needed(
        membership,
        config.start_date,
        config.end_date,
    )

    all_download_tickers = sorted(set(tickers_needed + [config.benchmark_ticker]))

    # ========================================================
    # 2. Datos de mercado con caché
    # ========================================================

    prices_raw, download_report, market_data_source = load_or_download_market_data(
        all_download_tickers,
        config,
    )

    run_metadata["market_data_source"] = market_data_source

    prices_raw.to_csv("data/raw/prices_raw.csv")
    download_report.to_csv("logs/download_report.csv", index=False)

    if config.benchmark_ticker not in prices_raw.columns:
        raise RuntimeError(
            f"No se pudo descargar/cargar el benchmark {config.benchmark_ticker}. "
            f"Prueba con '^GSPC' o revisa conexión/yfinance."
        )

    benchmark_prices = prices_raw[config.benchmark_ticker].dropna()

    asset_prices_raw = prices_raw.drop(columns=[config.benchmark_ticker], errors="ignore")
    asset_prices, exclusions = clean_prices(asset_prices_raw, config.min_price_coverage)

    asset_prices.to_csv("data/processed/prices_clean.csv")
    compute_simple_returns(asset_prices).to_csv("data/processed/returns.csv")
    exclusions.to_csv("logs/exclusions.csv", index=False)

    # ========================================================
    # 3. Backtest de estrategia
    # ========================================================

    if config.strategy_mode == "static_buy_hold":
        strategy_returns, strategy_equity, weights, strategy_meta = backtest_static_buy_hold(
            asset_prices,
            membership,
            config,
        )

    elif config.strategy_mode == "dynamic_equal_weight":
        strategy_returns, strategy_equity, weights, strategy_meta = backtest_dynamic_equal_weight(
            asset_prices,
            membership,
            config,
        )

    else:
        raise ValueError(f"strategy_mode no reconocido: {config.strategy_mode}")

    # ========================================================
    # 4. Benchmark
    # ========================================================

    benchmark_returns, benchmark_equity = backtest_benchmark(
        benchmark_prices,
        config,
    )

    common_dates = strategy_equity.index.intersection(benchmark_equity.index)

    strategy_equity = strategy_equity.loc[common_dates]
    strategy_returns = strategy_returns.loc[common_dates]

    benchmark_equity = benchmark_equity.loc[common_dates]
    benchmark_returns = benchmark_returns.loc[common_dates]

    # ========================================================
    # 5. Métricas principales
    # ========================================================

    strategy_metrics = compute_metrics(
        strategy_returns,
        strategy_equity,
        config,
        "SP100_strategy",
    )

    benchmark_metrics = compute_metrics(
        benchmark_returns,
        benchmark_equity,
        config,
        config.benchmark_name,
    )

    metrics = pd.DataFrame([strategy_metrics, benchmark_metrics])

    # ========================================================
    # 6. Turnover
    # ========================================================

    turnover = compute_turnover(weights)

    # ========================================================
    # 7. Sensibilidad a costes
    # ========================================================

    sensitivity_rows = []

    for cost in config.cost_sensitivity:
        if config.strategy_mode == "static_buy_hold":
            r_s, eq_s, _, _ = backtest_static_buy_hold(
                asset_prices,
                membership,
                config,
                transaction_cost=cost,
            )

        else:
            r_s, eq_s, _, _ = backtest_dynamic_equal_weight(
                asset_prices,
                membership,
                config,
                transaction_cost=cost,
            )

        eq_s = eq_s.loc[eq_s.index.intersection(common_dates)]
        r_s = r_s.loc[eq_s.index]

        m = compute_metrics(r_s, eq_s, config, f"strategy_cost_{cost}")
        m["transaction_cost"] = cost
        sensitivity_rows.append(m)

    sensitivity = pd.DataFrame(sensitivity_rows)

    # ========================================================
    # 8. Bootstrap simple
    # ========================================================

    boot = bootstrap_metrics_simple(
        strategy_returns,
        config,
        n_bootstrap=1000,
    )

    boot_summary = boot[["CAGR", "Sharpe", "Sortino", "max_drawdown", "Calmar"]].quantile(
        [0.025, 0.05, 0.50, 0.95, 0.975]
    )

    # ========================================================
    # 9. Exportar outputs
    # ========================================================

    strategy_returns.to_csv(out / "strategy_returns.csv")
    strategy_equity.to_csv(out / "strategy_equity.csv")

    benchmark_returns.to_csv(out / "benchmark_returns.csv")
    benchmark_equity.to_csv(out / "benchmark_equity.csv")

    weights.to_csv(out / "weights.csv")
    turnover.to_csv(out / "turnover.csv")

    metrics.to_csv(out / "metrics.csv", index=False)
    sensitivity.to_csv(out / "cost_sensitivity.csv", index=False)

    boot.to_csv(out / "bootstrap_metrics.csv", index=False)
    boot_summary.to_csv(out / "bootstrap_summary.csv")

    # ========================================================
    # 10. Metadata
    # ========================================================

    run_metadata["strategy_metadata"] = strategy_meta
    run_metadata["n_raw_assets_downloaded"] = len(tickers_needed)
    run_metadata["n_assets_after_cleaning"] = int(asset_prices.shape[1])
    run_metadata["n_excluded_assets"] = int(len(exclusions))
    run_metadata["benchmark_ticker"] = config.benchmark_ticker

    if "SURVIVORSHIP_BIASED" in membership_source:
        membership_warning = "WARNING: current/fallback constituents used. Survivorship bias present."
    elif "wikipedia" in membership_source.lower():
        membership_warning = (
            "Historical public reconstruction from Wikipedia page revisions. "
            "This avoids using future constituents, but is not the official licensed S&P DJI constituent history."
        )
    else:
        membership_warning = "Historical membership file used."

    run_metadata["membership_warning"] = membership_warning

    save_json(run_metadata, out / "metadata.json")

    # ========================================================
    # 11. Figuras
    # ========================================================

    plot_equity_curves(
        strategy_equity,
        benchmark_equity,
        out / "figures/equity_curves.png",
    )

    plot_drawdowns(
        strategy_equity,
        benchmark_equity,
        out / "figures/drawdowns.png",
    )

    # ========================================================
    # 12. Resumen por pantalla
    # ========================================================

    print("\n=== Bloque 0 completado ===")
    print(f"Modo estrategia: {config.strategy_mode}")
    print(f"Fuente membership: {membership_source}")
    print(f"Fuente datos mercado: {market_data_source}")
    print(f"Activos tras limpieza: {asset_prices.shape[1]}")
    print(f"Benchmark: {config.benchmark_ticker}")
    print(f"Outputs guardados en: {out.resolve()}")

    print("\nMétricas principales:")
    with pd.option_context("display.max_columns", None, "display.width", 160):
        print(metrics)


print("Bloque 2.5 cargado correctamente. run_block0 ahora usa caché de datos.")

"""Configuración"""


In [ ]:
# ============================================================
# cELDA 3 - CONFIGURACIÓN DEL EXPERIMENTO


In [ ]:
# ============================================================

from pathlib import Path

# Asegurar carpetas en Drive
Path(f"{TFG_BASE_DIR}").mkdir(parents=True, exist_ok=True)
Path(f"{TFG_BASE_DIR}/data").mkdir(parents=True, exist_ok=True)
Path(f"{TFG_BASE_DIR}/data/cache").mkdir(parents=True, exist_ok=True)
Path(f"{TFG_BASE_DIR}/results").mkdir(parents=True, exist_ok=True)
Path(f"{TFG_BASE_DIR}/logs").mkdir(parents=True, exist_ok=True)

CONFIG.start_date = "2021-01-01"
CONFIG.end_date = "2026-01-01"

# Benchmark S&P 500.
CONFIG.benchmark_ticker = "SPY"

# Usamos estrategia dinámica para respetar entradas/salidas del universo histórico.
CONFIG.strategy_mode = "dynamic_equal_weight"
CONFIG.rebalance_frequency = "M"

# Costes
CONFIG.transaction_cost = 0.001

# Tasa libre de riesgo anual provisional.
CONFIG.risk_free_rate_annual = 0.0225

# Reconstrucción histórica del S&P 100.
CONFIG.use_reconstructed_membership = True

# False = no reconstruye desde Wikipedia si ya existe el CSV reconstruido.
CONFIG.force_rebuild_membership = False

CONFIG.membership_history_frequency = "MS"

# Si falla la reconstrucción histórica, NO vuelve a lista actual.
CONFIG.allow_current_fallback = False

# Caché de precios.
CONFIG.use_cached_market_data = True

# False = no vuelve a descargar precios si ya existen en caché.
CONFIG.force_redownload_market_data = False

# Guardamos caché en Drive.
CONFIG.market_data_cache_dir = f"{TFG_BASE_DIR}/data/cache"

# Guardamos resultados en Drive.
CONFIG.output_dir = f"{TFG_BASE_DIR}/results/block0_sp100_survivorship_corrected"

# Guardamos membership histórico y logs en Drive.
CONFIG.reconstructed_membership_file = f"{TFG_BASE_DIR}/data/sp100_membership_reconstructed_wikipedia.csv"
CONFIG.snapshot_log_file = f"{TFG_BASE_DIR}/data/sp100_wikipedia_snapshot_log.csv"

CONFIG

"""Ejecutar"""

run_block0(CONFIG)

"""Resultados"""


In [ ]:
# ============================================================
# CELDA 5 - VER RESULTADOS PRINCIPALES


In [ ]:
# ============================================================

from pathlib import Path
import pandas as pd
import json

results_dir = Path(CONFIG.output_dir)

metrics_path = results_dir / "metrics.csv"
sensitivity_path = results_dir / "cost_sensitivity.csv"
bootstrap_summary_path = results_dir / "bootstrap_summary.csv"
metadata_path = results_dir / "metadata.json"

print(f"Carpeta de resultados: {results_dir}")

if not metrics_path.exists():
    raise FileNotFoundError(
        f"No existe {metrics_path}. Ejecuta primero el Bloque 4: run_block0(CONFIG)"
    )

metrics = pd.read_csv(metrics_path)
cost_sensitivity = pd.read_csv(sensitivity_path)
bootstrap_summary = pd.read_csv(bootstrap_summary_path, index_col=0)

with open(metadata_path, "r", encoding="utf-8") as f:
    metadata = json.load(f)

print("\n=== Metadata clave ===")
print("Fuente membership:", metadata.get("membership_source"))
print("Aviso membership:", metadata.get("membership_warning"))
print("Fuente datos mercado:", metadata.get("market_data_source"))
print("Benchmark:", metadata.get("benchmark_ticker"))
print("Activos tras limpieza:", metadata.get("n_assets_after_cleaning"))

print("\n=== Métricas principales ===")
display(metrics)

print("\n=== Sensibilidad a costes ===")
display(cost_sensitivity)

print("\n=== Bootstrap summary ===")
display(bootstrap_summary)

"""Figuras"""


In [ ]:
# ============================================================
# CELDA 6 - VER FIGURAS


In [ ]:
# ============================================================

from pathlib import Path
from IPython.display import Image, display

results_dir = Path(CONFIG.output_dir)
figures_dir = results_dir / "figures"

equity_path = figures_dir / "equity_curves.png"
drawdown_path = figures_dir / "drawdowns.png"

print(f"Carpeta de figuras: {figures_dir}")

if not equity_path.exists():
    raise FileNotFoundError(
        f"No existe {equity_path}. Ejecuta primero el Bloque 4: run_block0(CONFIG)"
    )

if not drawdown_path.exists():
    raise FileNotFoundError(
        f"No existe {drawdown_path}. Ejecuta primero el Bloque 4: run_block0(CONFIG)"
    )

print("\n=== Curva de capital ===")
display(Image(filename=str(equity_path)))

print("\n=== Drawdown ===")
display(Image(filename=str(drawdown_path)))

"""Descargar"""

!zip -r block0_results.zip results data logs > /dev/null

from google.colab import files
files.download("block0_results.zip")


In [ ]:
# ============================================================
# MODULO C5 - VERSION PUBLICA CENSURADA


In [ ]:
# ============================================================
# Este bloque sustituye la implementacion privada del modulo C5.
#
# No se incluye la logica interna de generacion de senales ni ninguna
# informacion que permita reconstruir la composicion interna de la
# estrategia.
#
# Esta version publica parte exclusivamente de series de rentabilidades
# observadas. A partir de dichas series se reproducen los analisis
# estadisticos utilizados en el trabajo:
#
# - curvas de capital;
# - metricas de rentabilidad y riesgo;
# - bootstrap temporal;
# - White's Reality Check studentizado;
# - regresion factorial opcional.


In [ ]:
# ============================================================

from pathlib import Path
import json
import math
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")


In [ ]:
# ============================================================
# C5 PUBLICO - CONFIGURACION


In [ ]:
# ============================================================

C5_PUBLIC_MODULE_NAME = "C5_public_returns_only"

C5_PUBLIC_ANALYSIS_START = "2021-01-01"
C5_PUBLIC_ANALYSIS_END = "2025-12-31"

C5_PUBLIC_SELECTED_SERIES = "C5_selected"

C5_PUBLIC_REFERENCE_COLUMNS = [
    "BuyHold",
    "Benchmark",
]

C5_PUBLIC_TRADING_DAYS_PER_YEAR = 252
C5_PUBLIC_RISK_FREE_RATE_ANNUAL = 0.0225

C5_PUBLIC_N_BOOTSTRAP = 1000
C5_PUBLIC_BLOCK_LENGTH = 20
C5_PUBLIC_RANDOM_SEED = 123

C5_PUBLIC_OUTPUT_DIR = Path("results/C5_public_returns_only")
C5_PUBLIC_TABLES_DIR = C5_PUBLIC_OUTPUT_DIR / "tables"
C5_PUBLIC_FIGURES_DIR = C5_PUBLIC_OUTPUT_DIR / "figures"
C5_PUBLIC_DATA_DIR = C5_PUBLIC_OUTPUT_DIR / "data"

for directory in [
    C5_PUBLIC_OUTPUT_DIR,
    C5_PUBLIC_TABLES_DIR,
    C5_PUBLIC_FIGURES_DIR,
    C5_PUBLIC_DATA_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)


In [ ]:
# ============================================================
# C5 PUBLICO - CARGA DE RENTABILIDADES


In [ ]:
# ============================================================

def c5_public_load_returns_table(path):
    """
    Carga una tabla de rentabilidades observadas.

    El archivo debe contener una columna de fechas y una o varias columnas
    numericas de rentabilidad.
    """

    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(f"No existe el archivo de rentabilidades: {path}")

    df = pd.read_csv(path)

    date_col = None

    for candidate in ["Date", "date", "DATE", "timestamp", "Timestamp"]:
        if candidate in df.columns:
            date_col = candidate
            break

    if date_col is None:
        date_col = df.columns[0]

    df[date_col] = pd.to_datetime(df[date_col])
    df = df.set_index(date_col).sort_index()

    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df = df.replace([np.inf, -np.inf], np.nan)

    return df


def c5_public_restrict_period(returns_table, start_date, end_date):
    """
    Recorta una tabla de rentabilidades al periodo analizado.
    """

    out = returns_table.copy()

    out = out.loc[
        (out.index >= pd.Timestamp(start_date)) &
        (out.index <= pd.Timestamp(end_date))
    ]

    out = out.dropna(axis=0, how="all")

    return out


def c5_public_equity_curve(returns):
    """
    Construye una curva de capital normalizada.
    """

    r = pd.Series(returns).replace([np.inf, -np.inf], np.nan).fillna(0.0)

    return (1.0 + r).cumprod()


In [ ]:
# ============================================================
# C5 PUBLICO - METRICAS


In [ ]:
# ============================================================

def c5_public_max_drawdown(equity):
    """
    Calcula el maximo drawdown de una curva de capital.
    """

    e = pd.Series(equity).dropna()

    if len(e) == 0:
        return np.nan

    running_max = e.cummax()
    drawdown = e / running_max - 1.0

    return float(drawdown.min())


def c5_public_compute_metrics(
    returns,
    label,
    risk_free_rate_annual=C5_PUBLIC_RISK_FREE_RATE_ANNUAL,
    trading_days=C5_PUBLIC_TRADING_DAYS_PER_YEAR,
):
    """
    Calcula metricas de rentabilidad y riesgo.
    """

    r = pd.Series(returns).dropna().astype(float)

    if len(r) == 0:
        return {
            "series": label,
            "n_observations": 0,
            "cumulative_return": np.nan,
            "CAGR": np.nan,
            "annualized_volatility": np.nan,
            "Sharpe": np.nan,
            "Sortino": np.nan,
            "max_drawdown": np.nan,
            "Calmar": np.nan,
            "hit_ratio": np.nan,
        }

    eq = c5_public_equity_curve(r)

    n_obs = len(r)
    years = n_obs / trading_days

    cumulative_return = float(eq.iloc[-1] - 1.0)

    if years > 0 and eq.iloc[-1] > 0:
        cagr = float(eq.iloc[-1] ** (1.0 / years) - 1.0)
    else:
        cagr = np.nan

    annualized_volatility = float(r.std(ddof=1) * math.sqrt(trading_days))

    rf_daily = (1.0 + risk_free_rate_annual) ** (1.0 / trading_days) - 1.0
    excess = r - rf_daily

    if len(excess) > 1 and excess.std(ddof=1) > 0:
        sharpe = float(excess.mean() / excess.std(ddof=1) * math.sqrt(trading_days))
    else:
        sharpe = np.nan

    downside = excess[excess < 0]

    if len(downside) > 1 and downside.std(ddof=1) > 0:
        sortino = float(excess.mean() / downside.std(ddof=1) * math.sqrt(trading_days))
    else:
        sortino = np.nan

    mdd = c5_public_max_drawdown(eq)

    if pd.notna(mdd) and mdd < 0:
        calmar = float(cagr / abs(mdd))
    else:
        calmar = np.nan

    hit_ratio = float((r > 0).mean())

    return {
        "series": label,
        "n_observations": int(n_obs),
        "cumulative_return": cumulative_return,
        "CAGR": cagr,
        "annualized_volatility": annualized_volatility,
        "Sharpe": sharpe,
        "Sortino": sortino,
        "max_drawdown": mdd,
        "Calmar": calmar,
        "hit_ratio": hit_ratio,
    }


def c5_public_compute_metrics_table(returns_table):
    """
    Calcula metricas para todas las columnas de una tabla de rentabilidades.
    """

    rows = []

    for col in returns_table.columns:
        s = returns_table[col].dropna()

        if len(s) == 0:
            continue

        rows.append(
            c5_public_compute_metrics(
                returns=s,
                label=col,
            )
        )

    return pd.DataFrame(rows)


In [ ]:
# ============================================================
# C5 PUBLICO - FIGURAS


In [ ]:
# ============================================================

def c5_public_plot_equity_curves(returns_table, columns, output_path):
    """
    Dibuja curvas de capital normalizadas.
    """

    plt.figure(figsize=(10, 6))

    for col in columns:
        if col not in returns_table.columns:
            continue

        eq = c5_public_equity_curve(returns_table[col])
        eq.plot(label=col)

    plt.title("C5 - Curvas de capital")
    plt.xlabel("Fecha")
    plt.ylabel("Capital normalizado")
    plt.legend()
    plt.tight_layout()
    plt.savefig(output_path, dpi=160)
    plt.close()


def c5_public_plot_drawdowns(returns_table, columns, output_path):
    """
    Dibuja drawdowns.
    """

    plt.figure(figsize=(10, 6))

    for col in columns:
        if col not in returns_table.columns:
            continue

        eq = c5_public_equity_curve(returns_table[col])
        dd = eq / eq.cummax() - 1.0
        dd.plot(label=col)

    plt.title("C5 - Drawdowns")
    plt.xlabel("Fecha")
    plt.ylabel("Drawdown")
    plt.legend()
    plt.tight_layout()
    plt.savefig(output_path, dpi=160)
    plt.close()


In [ ]:
# ============================================================
# C5 PUBLICO - BOOTSTRAP TEMPORAL


In [ ]:
# ============================================================

def c5_public_block_bootstrap_samples(
    returns,
    n_bootstrap=C5_PUBLIC_N_BOOTSTRAP,
    block_length=C5_PUBLIC_BLOCK_LENGTH,
    random_seed=C5_PUBLIC_RANDOM_SEED,
):
    """
    Genera muestras bootstrap por bloques.
    """

    rng = np.random.default_rng(random_seed)

    r = pd.Series(returns).dropna().astype(float).to_numpy()
    n = len(r)

    if n == 0:
        raise ValueError("La serie de rentabilidades esta vacia.")

    samples = []

    for _ in range(n_bootstrap):
        boot = []

        while len(boot) < n:
            start = rng.integers(0, max(1, n - block_length + 1))
            block = r[start:start + block_length]
            boot.extend(block.tolist())

        samples.append(np.array(boot[:n]))

    return samples


def c5_public_bootstrap_equity_percentiles(
    returns,
    n_bootstrap=C5_PUBLIC_N_BOOTSTRAP,
    block_length=C5_PUBLIC_BLOCK_LENGTH,
    random_seed=C5_PUBLIC_RANDOM_SEED,
):
    """
    Calcula percentiles bootstrap de curvas de capital.
    """

    samples = c5_public_block_bootstrap_samples(
        returns=returns,
        n_bootstrap=n_bootstrap,
        block_length=block_length,
        random_seed=random_seed,
    )

    curves = []

    for sample in samples:
        curves.append(np.cumprod(1.0 + sample))

    matrix = np.vstack(curves)

    percentiles = pd.DataFrame({
        "p2_5": np.percentile(matrix, 2.5, axis=0),
        "p50": np.percentile(matrix, 50.0, axis=0),
        "p97_5": np.percentile(matrix, 97.5, axis=0),
    })

    return percentiles


def c5_public_plot_bootstrap_percentiles(percentiles, output_path):
    """
    Dibuja percentiles bootstrap de capital.
    """

    x = np.arange(len(percentiles))

    plt.figure(figsize=(10, 6))
    plt.plot(x, percentiles["p50"], label="Mediana")
    plt.fill_between(
        x,
        percentiles["p2_5"],
        percentiles["p97_5"],
        alpha=0.25,
        label="Intervalo 2,5%-97,5%",
    )
    plt.title("C5 - Bootstrap temporal")
    plt.xlabel("Sesion")
    plt.ylabel("Capital normalizado")
    plt.legend()
    plt.tight_layout()
    plt.savefig(output_path, dpi=160)
    plt.close()


In [ ]:
# ============================================================
# C5 PUBLICO - WHITE'S REALITY CHECK STUDENTIZADO


In [ ]:
# ============================================================

def c5_public_studentized_statistic(differential_returns):
    """
    Calcula un estadistico tipo t sobre diferenciales de rentabilidad.
    """

    d = pd.Series(differential_returns).dropna().astype(float)

    if len(d) < 2:
        return np.nan

    std = d.std(ddof=1)

    if std <= 0 or pd.isna(std):
        return np.nan

    return float(d.mean() / (std / np.sqrt(len(d))))


def c5_public_studentized_reality_check(
    candidate_returns,
    reference_returns,
    n_bootstrap=C5_PUBLIC_N_BOOTSTRAP,
    block_length=C5_PUBLIC_BLOCK_LENGTH,
    random_seed=C5_PUBLIC_RANDOM_SEED,
):
    """
    Aplica White's Reality Check studentizado a partir de rentabilidades.
    """

    rng = np.random.default_rng(random_seed)

    aligned = candidate_returns.join(
        reference_returns.rename("reference"),
        how="inner",
    ).dropna()

    candidates = [c for c in aligned.columns if c != "reference"]

    differentials = aligned[candidates].subtract(aligned["reference"], axis=0)

    observed_stats = differentials.apply(
        c5_public_studentized_statistic,
        axis=0,
    )

    observed_max = observed_stats.max()

    centered = differentials - differentials.mean(axis=0)

    n = len(centered)
    values = centered.to_numpy()

    boot_max_stats = []

    for _ in range(n_bootstrap):
        boot_indices = []

        while len(boot_indices) < n:
            start = rng.integers(0, max(1, n - block_length + 1))
            boot_indices.extend(range(start, min(start + block_length, n)))

        boot_indices = boot_indices[:n]

        sample = pd.DataFrame(
            values[boot_indices, :],
            columns=candidates,
        )

        boot_stats = sample.apply(
            c5_public_studentized_statistic,
            axis=0,
        )

        boot_max_stats.append(boot_stats.max())

    boot_max_stats = np.array(boot_max_stats)

    p_value = float(np.mean(boot_max_stats >= observed_max))

    summary = pd.DataFrame({
        "candidate": observed_stats.index,
        "t_stat": observed_stats.values,
    }).sort_values("t_stat", ascending=False)

    return {
        "observed_max_t": float(observed_max),
        "p_value_corrected": p_value,
        "best_candidate_by_t": str(summary.iloc[0]["candidate"]),
        "summary": summary,
    }


In [ ]:
# ============================================================
# C5 PUBLICO - REGRESION FACTORIAL OPCIONAL


In [ ]:
# ============================================================

def c5_public_run_factor_regression(strategy_returns, factor_returns):
    """
    Regresion lineal de una serie de rentabilidades sobre factores externos.
    """

    import statsmodels.api as sm

    data = factor_returns.copy()
    data["strategy"] = strategy_returns

    data = data.dropna()

    y = data["strategy"]
    X = data.drop(columns=["strategy"])

    X = sm.add_constant(X)

    model = sm.OLS(y, X).fit()

    result = pd.DataFrame({
        "factor": model.params.index,
        "coefficient": model.params.values,
        "t_stat": model.tvalues.values,
        "p_value": model.pvalues.values,
    })

    return model, result


In [ ]:
# ============================================================
# C5 PUBLICO - PIPELINE


In [ ]:
# ============================================================

def run_c5_public(
    returns_csv_path,
    factor_csv_path=None,
):
    """
    Ejecuta la version publica censurada del modulo C5.
    """

    returns_table = c5_public_load_returns_table(returns_csv_path)

    returns_table = c5_public_restrict_period(
        returns_table,
        C5_PUBLIC_ANALYSIS_START,
        C5_PUBLIC_ANALYSIS_END,
    )

    returns_table.to_csv(
        C5_PUBLIC_DATA_DIR / "C5_public_returns_table.csv",
    )

    if C5_PUBLIC_SELECTED_SERIES not in returns_table.columns:
        raise KeyError(
            f"No se encuentra la columna {C5_PUBLIC_SELECTED_SERIES} "
            "en la tabla de rentabilidades."
        )

    metrics = c5_public_compute_metrics_table(returns_table)

    metrics.to_csv(
        C5_PUBLIC_TABLES_DIR / "C5_public_metrics.csv",
        index=False,
    )

    display_columns = [
        col for col in [C5_PUBLIC_SELECTED_SERIES] + C5_PUBLIC_REFERENCE_COLUMNS
        if col in returns_table.columns
    ]

    c5_public_plot_equity_curves(
        returns_table,
        display_columns,
        C5_PUBLIC_FIGURES_DIR / "C5_public_equity_curves.png",
    )

    c5_public_plot_drawdowns(
        returns_table,
        display_columns,
        C5_PUBLIC_FIGURES_DIR / "C5_public_drawdowns.png",
    )

    bootstrap_percentiles = c5_public_bootstrap_equity_percentiles(
        returns=returns_table[C5_PUBLIC_SELECTED_SERIES],
        n_bootstrap=C5_PUBLIC_N_BOOTSTRAP,
        block_length=C5_PUBLIC_BLOCK_LENGTH,
        random_seed=C5_PUBLIC_RANDOM_SEED,
    )

    bootstrap_percentiles.to_csv(
        C5_PUBLIC_TABLES_DIR / "C5_public_bootstrap_percentiles.csv",
        index=False,
    )

    c5_public_plot_bootstrap_percentiles(
        bootstrap_percentiles,
        C5_PUBLIC_FIGURES_DIR / "C5_public_bootstrap_percentiles.png",
    )

    candidate_columns = [
        col for col in returns_table.columns
        if col.startswith("C5_candidate_")
    ]

    wrc_results = {}

    if len(candidate_columns) > 0:
        candidate_returns = returns_table[candidate_columns].copy()

        for reference_column in C5_PUBLIC_REFERENCE_COLUMNS:
            if reference_column not in returns_table.columns:
                continue

            result = c5_public_studentized_reality_check(
                candidate_returns=candidate_returns,
                reference_returns=returns_table[reference_column],
                n_bootstrap=C5_PUBLIC_N_BOOTSTRAP,
                block_length=C5_PUBLIC_BLOCK_LENGTH,
                random_seed=C5_PUBLIC_RANDOM_SEED,
            )

            safe_reference = (
                reference_column
                .replace("&", "and")
                .replace(" ", "_")
            )

            result["summary"].to_csv(
                C5_PUBLIC_TABLES_DIR / f"C5_public_wrc_vs_{safe_reference}.csv",
                index=False,
            )

            wrc_results[reference_column] = {
                "observed_max_t": result["observed_max_t"],
                "p_value_corrected": result["p_value_corrected"],
                "best_candidate_by_t": result["best_candidate_by_t"],
            }

    factor_results_path = None

    if factor_csv_path is not None:
        factor_returns = pd.read_csv(
            factor_csv_path,
            index_col=0,
            parse_dates=True,
        )

        model, factor_results = c5_public_run_factor_regression(
            strategy_returns=returns_table[C5_PUBLIC_SELECTED_SERIES],
            factor_returns=factor_returns,
        )

        factor_results_path = C5_PUBLIC_TABLES_DIR / "C5_public_factor_regression.csv"
        factor_results.to_csv(factor_results_path, index=False)

    metadata = {
        "module": C5_PUBLIC_MODULE_NAME,
        "analysis_start": C5_PUBLIC_ANALYSIS_START,
        "analysis_end": C5_PUBLIC_ANALYSIS_END,
        "selected_series": C5_PUBLIC_SELECTED_SERIES,
        "source": "public_returns_only",
        "description": (
            "Version publica censurada. El modulo parte exclusivamente de "
            "series de rentabilidades observadas y no incluye informacion "
            "interna de la estrategia."
        ),
        "metrics_path": str(C5_PUBLIC_TABLES_DIR / "C5_public_metrics.csv"),
        "bootstrap_path": str(C5_PUBLIC_TABLES_DIR / "C5_public_bootstrap_percentiles.csv"),
        "wrc_results": wrc_results,
        "factor_results_path": str(factor_results_path) if factor_results_path is not None else None,
    }

    with open(C5_PUBLIC_OUTPUT_DIR / "C5_public_metadata.json", "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=4, ensure_ascii=False)

    print("Modulo C5 publico ejecutado.")
    print("Rentabilidades:", returns_csv_path)
    print("Metricas:", C5_PUBLIC_TABLES_DIR / "C5_public_metrics.csv")
    print("Figuras:", C5_PUBLIC_FIGURES_DIR)

    return {
        "returns_table": returns_table,
        "metrics": metrics,
        "bootstrap_percentiles": bootstrap_percentiles,
        "wrc_results": wrc_results,
        "metadata": metadata,
    }


In [ ]:
# ============================================================
# EJEMPLO DE USO


In [ ]:
# ============================================================
# results = run_c5_public(
#     returns_csv_path="data/C5_public_returns.csv",
#     factor_csv_path=None,
# )
